# Spatial TX registration to blockface

Process
1. Register spatial TX barcode to its respective blockface image
    a. If blockface section doesn't exist, pick the closets section  
    b. If blockface set doesn't exist, duplicate image from closest set and rename to target set. (See readme for mapping)  
    c. Perform QC on registration from the outputted QC images  
    d. For registration that failed, this is due to a failure in the initial rotation search. A rotation is visually picked and the registration is re-run


In [7]:
from hmba_stx_registration import Specimen, get_metadata_df, register_cells_to_blockface
from pathlib import Path
donor = "H24.30.005"
base_path = Path("/home/mike/workspace/data/human_brain/") / donor
sptx_path = base_path / "sptx"
barcodes_path = sptx_path / "barcodes"
um_per_px = 20
mm_per_px = um_per_px / 1000
metadata_df = get_metadata_df(barcodes_path)
mapping_date = "20260529"
output_date = "20260615"
table_label = "MTL_subclass_label"

/home/mike/workspace/data/human_brain/H24.30.005/sptx/barcodes/barcode_csv/barcode_csv_specimen_metadata.json does not exist, skipping


### Perform registration between ST and Blockface

For each barcode:
-> Register to blockface  
**Inputs:**  
1. blockface image path  
2. Specimen object  
    a. cells table  
    b. specimen metadata  

**Outputs:**  
1. transforms_out_path: directory to store   affines in .npy format  
2. qc_path: directory to store plots to visualize registration for QC  


In [26]:
barcodes = metadata_df['barcode'].tolist()
transforms_path_existing = sptx_path / "sptx_to_bf_transforms"
existing_specimens = [p.stem.split("_")[0] for p in transforms_path_existing.glob("*.npy")]
transforms_path = sptx_path / f"st2block_output_{output_date}"
existing_specimens.extend([p.stem.split("_")[0] for p in transforms_path.glob("*.npy")])
for barcode in barcodes:
    specimen = Specimen(barcode, barcodes_path, date=mapping_date)
    if specimen.specimen_name in existing_specimens:
        print(f"Specimen {specimen.specimen_name} already has a transform, skipping.")
        continue
    print("Registering specimen:", specimen.specimen_name)
    bf_img_path_base = base_path / 'blocks' / specimen.slab_name / 'processed'
    try:
        bf_img_path = list(bf_img_path_base.glob(f"{specimen.specimen_set_name}*.png"))[0]
    except IndexError:
        print(f"No blockface image found for specimen {specimen.specimen_name}, skipping.")
        continue
    st2bf_affine = register_cells_to_blockface(specimen.cells_table,
                            bf_img_path,
                            specimen.specimen_name,
                            transforms_out_path=transforms_path,
                            qc_path=transforms_path / "qc",
                            table_label=table_label,
                            gamma=0.3
                            )

Specimen H24.30.005.CX.25.02.03.02 already has a transform, skipping.
Specimen H24.30.005.CX.23.01.05.02 already has a transform, skipping.
Specimen H24.30.005.CX.23.01.03.02 already has a transform, skipping.
Specimen H24.30.005.CX.29.02.03.02 already has a transform, skipping.
Specimen H24.30.005.CX.27.02.05.02 already has a transform, skipping.
Specimen H24.30.005.CX.25.03.05.01 already has a transform, skipping.
Specimen H24.30.005.CX.27.02.03.02 already has a transform, skipping.
Specimen H24.30.005.CX.25.03.05.02 already has a transform, skipping.
Specimen H24.30.005.CX.29.01.05.02 already has a transform, skipping.
Specimen H24.30.005.CX.25.02.01.02 already has a transform, skipping.
Registering specimen: H24.30.005.CX.23.02.05.02


INFO:hmba_stx_registration.st_to_block:Best rotation: 165°, shift: [ -2. -26.], corr: 0.8752
INFO:hmba_stx_registration.st_to_block:Saved blockface registration for H24.30.005.CX.23.02.05.02


Specimen H24.30.005.CX.29.01.03.02 already has a transform, skipping.
Specimen H24.30.005.CX.25.03.03.02 already has a transform, skipping.
Specimen H24.30.005.CX.27.01.05.02 already has a transform, skipping.
Registering specimen: H24.30.005.CX.23.02.03.02


INFO:hmba_stx_registration.st_to_block:Best rotation: 180°, shift: [ 62. -35.], corr: 0.7924
INFO:hmba_stx_registration.st_to_block:Saved blockface registration for H24.30.005.CX.23.02.03.02


Specimen H24.30.005.CX.25.02.05.02 already has a transform, skipping.
Specimen H24.30.005.CX.29.02.05.03 already has a transform, skipping.


### Rerun registration for sets that failed QC by visually defining the rotation

In [ ]:
setname_rotation = {
    'H24.30.005.CX.25.02.05': 180,
}

transforms_path = sptx_path / f"st2block_output_{output_date}"
for setname, rotation in setname_rotation.items():
    for barcode in metadata_df.loc[metadata_df.specimen_set_name == setname].barcode.values:
            specimen = Specimen(barcode, barcodes_path, date=mapping_date)
            print("Registering specimen:", specimen.specimen_name)
            bf_img_path_base = base_path / 'blocks' / specimen.slab_name / 'processed'
            bf_img_path = list(bf_img_path_base.glob(f"{specimen.specimen_set_name}*.png"))[0]

            st2bf_affine = register_cells_to_blockface(specimen.cells_table,
                                    bf_img_path,
                                    specimen.specimen_name,
                                    transforms_out_path=transforms_path,
                                    qc_path=transforms_path / "qc",
                                    gamma=0.3,
                                    rotation=rotation,
                                    table_label=table_label,
                                    )

Registering specimen: H24.30.005.CX.25.02.05.02


Cells table contains 264 rows with NaN center_x or center_y. These will be dropped for downstream processing.


## Registration from STx to slab using transforms from SVG

1. Obtain block2slab transforms  
    a. In this case, it was performed with SVG
1. Create dict of slab_id to slab_image


In [28]:
from hmba_stx_registration import Specimen, process_barcode
from hmba_stx_registration.st_to_slab import get_bf_affines_from_svg
from pathlib import Path
import matplotlib.pyplot as plt
from datetime import datetime
import logging
import os

os.makedirs("logs", exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    handlers=[
        logging.FileHandler("logs/st_to_slab_registration.log"),
        logging.StreamHandler(),
    ],
)

slab_ids = [23, 25, 27, 29]

bf_affines = {}
slab_imgs = {}
for slab_id in slab_ids:
    slab_imgs[slab_id] = plt.imread(base_path / "slabs" / "fresh" / "processed_20um_per_px" / f"CX{slab_id}.png")
    svg_path = base_path / f"{donor}.CX.{slab_id}.svg"
    if not svg_path.exists():
        print(f"Missing SVG: {svg_path}")
        continue
    bf_affines_slab = get_bf_affines_from_svg(svg_path, base_path)
    bf_affines.update(bf_affines_slab)

# save bf_affines to json
import json
with open(base_path / f"bf_affines_{output_date}.json", "w") as f:
    json.dump({k: v.tolist() for k, v in bf_affines.items()}, f)

barcodes = metadata_df['barcode'].tolist()
for barcode in barcodes:
    specimen = Specimen(barcode, barcodes_path, date=mapping_date)
    process_barcode(
        specimen,
        bf_affines,
        slab_imgs,
        transforms_path,
        table_label,
        um_per_px,
        mapping_date,
        output_date,
        version="0.3",
        sync_to_s3=True,
        sync_dryrun=False,
    )

INFO:hmba_stx_registration.st_to_slab:[1444202702] Copied block QC → H24.30.005.CX.25.02.03.02_registration_block_qc_20260615.png
INFO:hmba_stx_registration.st_to_slab:[1444202702] Saved transform manifest: H24.30.005.CX.25.02.03.02_coarse_transform_to_slab_mm_20260615.json
INFO:hmba_stx_registration.st_to_slab:[1444202702] Saved run manifest: H24.30.005.CX.25.02.03.02_run_manifest_20260615.json
INFO:hmba_stx_registration.utils:Running: aws s3 sync /home/mike/workspace/data/human_brain/H24.30.005/sptx/barcodes/1444202702/registration_results_20260615 s3://hmba-macaque-wg-802451596237-us-west-2/hmba_aim_2/Xenium/QM24.50.002/1444202702/registration_results_20260615 --profile storage --only-show-errors
INFO:hmba_stx_registration.st_to_slab:[1444202702] Synced to S3
INFO:hmba_stx_registration.st_to_slab:[1444202702] Done (H24.30.005.CX.25.02.03.02)
INFO:hmba_stx_registration.st_to_slab:[1451317976] Copied block QC → H24.30.005.CX.23.01.05.02_registration_block_qc_20260615.png
INFO:hmba_stx

## Generate QC plots for each plane

Visualize mosaicked STx sections for each plane



In [29]:
from hmba_stx_registration import apply_slab_transforms, plot_single_slab_scatter
import pandas as pd

barcodes_df = pd.read_csv(barcodes_path / f"barcode_csv/{donor}_section_metadata.csv", index_col=0)
figs_path = barcodes_path / "registration_plots" / f"registration_results_{output_date}"
figs_path.mkdir(parents=True, exist_ok=True)


for (slab, plane), group in barcodes_df.groupby(["slab", "slab_plane"], sort=True):
    bc_list = group["barcode"].tolist()
    print(f"Processing slab {slab} / plane {plane} with {sorted(group.block.to_list())} blocks")
    df = apply_slab_transforms(barcodes_path, registration_results_date=output_date, barcodes=bc_list, mapping_date=mapping_date)
    if df.empty:
        print(f"Slab {slab} / {plane}: no data, skipping")
        continue
    fig = plot_single_slab_scatter(
        df,
        label_col="MTL_neighborhood_name",
        exclude_labels=['Non-neuronal'],
        title=f"Slab {slab} – plane {plane}",
    )
    fig.savefig(figs_path / f"{donor}.CX.{int(slab)}.{plane}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Slab {slab} / {plane}: {len(df)} cells, {len(bc_list)} barcodes")

Processing slab 23 / plane Plane03 with [1, 2] blocks


INFO:hmba_stx_registration:[1451317597] Appended 120652 rows
INFO:hmba_stx_registration:[1451319097] Appended 73302 rows


Slab 23 / Plane03: 193954 cells, 2 barcodes
Processing slab 23 / plane Plane05 with [1, 2] blocks


INFO:hmba_stx_registration:[1451317976] Appended 127943 rows
INFO:hmba_stx_registration:[1451319367] Appended 89541 rows


Slab 23 / Plane05: 217484 cells, 2 barcodes
Processing slab 25 / plane Plane01 with [2] blocks


...

Slab 27 / Plane05: 196826 cells, 2 barcodes
Processing slab 29 / plane Plane03 with [1, 2] blocks


INFO:hmba_stx_registration:[1454345403] Appended 92455 rows
INFO:hmba_stx_registration:[1454335355] Appended 114038 rows


Slab 29 / Plane03: 206493 cells, 2 barcodes
Processing slab 29 / plane Plane05 with [1, 2] blocks


INFO:hmba_stx_registration:[1454335675] Appended 78313 rows
INFO:hmba_stx_registration:[1454345790] Appended 99947 rows


Slab 29 / Plane05: 178260 cells, 2 barcodes


In [30]:
# Sync results to S3

from hmba_stx_registration.utils import sync_dir_to_s3

figs_path = barcodes_path / "registration_plots"
sync_dir_to_s3(
    local_dir=figs_path,
    bucket="hmba-human-wg-802451596237-us-west-2",
    s3_dir=f"HPF_MEC/{donor}/registration_plots",
    profile="storage",
    dryrun=False,
    delete=False,
)

INFO:hmba_stx_registration.utils:Running: aws s3 sync /home/mike/workspace/data/human_brain/H24.30.005/sptx/barcodes/registration_plots s3://hmba-human-wg-802451596237-us-west-2/HPF_MEC/H24.30.005/registration_plots --profile storage --only-show-errors
